# HCR-Net — full architecture of the main model (SCR_HR)

**Paper title:** *Lightweight Single-Image Dehazing with Haze-Guided Kernel Routing*
**Model:** HCR-Net, preset **MHR** (run name SCR_HR): MAX_HY layout, 25 HCR blocks, HR router in every block, trained from scratch.

| | Value |
|---|---|
| Trainable parameters | **978,667** (limit 1,000,000) |
| Conv GMACs @ 256×256 | 6.936 |
| Levels | 4 (1, 1/2, 1/4, 1/8 resolution) |
| Widths | 24 / 48 / 96 / 136 |
| Blocks per stage (enc0, enc1, enc2, mid, dec2, dec1, dec0) | 4, 5, 2, 3, 2, 5, 4 = 25 |
| Attention | none (attention-free) |
| RESIDE-6K test (1,000 images, full resolution, last-epoch EMA) | **31.2128 dB / 0.9763** (indoor 29.579, outdoor 32.846) |
| Uneven-haze test (same 1,000 images, patchy haze) | 27.9744 dB / 0.9623 |

Parameter counts below come from rebuilding the layout: total 978,667 and static twin 956,544, the same as the training logs.

---

## 1. End-to-end data flow

```
I (B,3,H,W) in [0,1], H and W multiples of 8 (reflect-pad at test, crop back)
│
├─► haze_cues(I) ──► C = [dark channel, 4·local contrast, luminance]   (3 ch, 0 params)
│
[I, C] ──► stem: 3×3 conv, 6 → 24
│
ENCODER
  level 0  (24 ch,  H×W)     : 4 HCR blocks ──► skip0 ──► 2×2 conv stride 2 (24 → 48)
  level 1  (48 ch,  H/2)     : 5 HCR blocks ──► skip1 ──► 2×2 conv stride 2 (48 → 96)
  level 2  (96 ch,  H/4)     : 2 HCR blocks ──► skip2 ──► 2×2 conv stride 2 (96 → 136)
MIDDLE
  level 3  (136 ch, H/8)     : 3 HCR blocks
DECODER
  up: 1×1 conv (136 → 4·96) + PixelShuffle(2) ──► SKFusion(up, skip2) ──► 2 HCR blocks (96 ch)
  up: 1×1 conv (96  → 4·48) + PixelShuffle(2) ──► SKFusion(up, skip1) ──► 5 HCR blocks (48 ch)
  up: 1×1 conv (48  → 4·24) + PixelShuffle(2) ──► SKFusion(up, skip0) ──► 4 HCR blocks (24 ch)
HEAD
  3×3 conv (24 → 4) = [K (1 ch), B (3 ch)]
  J = I + K·I + B                      (soft reconstruction; clamp to [0,1] only at test)
```

Every stage pools the cues C to its own resolution and turns them into 6 **heterogeneity cues**. All blocks of that stage share them in their routers.

---

## 2. Physical haze cues (no parameters)

Computed once from the hazy input, in fp32, without gradient.

| Cue | Formula | Meaning |
|---|---|---|
| D, dark channel | min over RGB, then 7×7 min-filter | high = haze / bright veil |
| C, local contrast | 4 × (std in a 5×5 window, mean over RGB) | low = hazy |
| Lum, luminance | 0.299 R + 0.587 G + 0.114 B | brightness |

**Heterogeneity cues per stage** (C pooled to the stage size with adaptive average pooling, then 6 channels):

```
het = [ D,
        C,
        8 · (|∂x D| + |∂y D|),          # where the haze changes
        4 · (D   − box9(D)),            # local deviation of haze
        4 · (C   − box9(C)),            # local deviation of contrast
        4 · (Lum − box9(Lum)) ]         # local deviation of brightness
```

box9 is a 9×9 mean with reflect padding.

---

## 3. The HCR block (Haze-guided Context Routing block)

Input x (B, c, h, w). There are two residual halves, each wrapped by RescaleNorm.

```
u, s1, b1 = RescaleNorm1(x)
v  = pw_in(u)                                   1×1 conv, c → c
L  = DW3×3(v)                                   local mixer, 3-px span
G  = DW5×5, dilation 3 (v)                      context mixer, 13-px span
w  = sigmoid( prior_g + HR([u, het]) )          per pixel, one weight per channel group (4 groups)
m  = w · L + (1 − w) · G                        per group
x  = x + pw_out(m) · s1 + b1                    1×1 conv, rescale/re-bias, residual

u2, s2, b2 = RescaleNorm2(x)
x  = x + FFN(u2) · s2 + b2                      FFN = 1×1 (c → 2c) → ReLU → 1×1 (2c → c)
```

### 3.1 RescaleNorm (from DehazeFormer's RLN)
- It normalises each sample over (C, H, W): u = (x − μ)/σ · weight + bias.
- It also returns s = meta_s(σ) and b = meta_m(μ), two 1×1 convs from 1 channel to c. meta_s has bias 1 and meta_m has bias 0 at init.
- The block puts the removed brightness and contrast back after the branch, as `f(u)·s + b`.

### 3.2 HR router (the contribution)
| Item | Value |
|---|---|
| Input | block features u (c channels) + 6 heterogeneity cues |
| Layers | 1×1 conv (c + 6 → h) → GELU → depthwise 3×3 → GELU → 1×1 conv (h → 4) |
| Hidden width h | max(8, c/8): 8 at 24 and 48 ch, 12 at 96 ch, 17 at 136 ch |
| Output | 4 logits, one per channel group, added to a learned **static prior** per group |
| Gate | w_local = sigmoid(prior + correction); w_context = 1 − w_local |
| Precision | fp32 (outside AMP) |
| Init | last layer zero, prior 0, so the block starts as a fixed 0.5 / 0.5 mix |
| Global pooling | none (no train/test statistics mismatch) |

- **Static twin (HT):** the same block with the prior only, which is a learned fixed share per group. It has 956,544 params.
- HR − HT = 22,123 parameters (+2.3 %).

### 3.3 Per-block parameter cost
| Width c | Params per block | of which router + 4 priors |
|---|---|---|
| 24 | 5,096 | 368 |
| 48 | 16,928 | 560 |
| 96 | 61,796 | 1,412 |
| 136 | 120,861 | 2,677 |

---

## 4. Stages and parameter budget

| Part | Resolution | Width | Blocks | Params |
|---|---|---|---|---|
| Stem (3×3, 6 → 24) | 1 | 24 | — | 1,320 |
| Encoder level 0 | 1 | 24 | 4 | 20,384 |
| Encoder level 1 | 1/2 | 48 | 5 | 84,640 |
| Encoder level 2 | 1/4 | 96 | 2 | 123,592 |
| Middle | 1/8 | 136 | 3 | 362,583 |
| Decoder level 2 | 1/4 | 96 | 2 | 123,592 |
| Decoder level 1 | 1/2 | 48 | 5 | 84,640 |
| Decoder level 0 | 1 | 24 | 4 | 20,384 |
| Downsampling (3 × 2×2 stride-2 conv) | | | | 75,544 |
| Upsampling (3 × 1×1 conv + PixelShuffle) | | | | 75,936 |
| SK fusion (3) | | | | 5,184 |
| Head (3×3, 24 → 4) | | | | 868 |
| **Total** | | | **25** | **978,667** |

All HR routers together: 22,123 weights. The 100 static priors (4 per block) are also in the static twin.

---

## 5. SK fusion (skip connections, from DehazeFormer)
- s = GAP(up + skip) → 1×1 (c → max(c/8, 8), no bias) → GELU → 1×1 (→ 2c, no bias).
- softmax over the 2 inputs, per channel → out = w₀·up + w₁·skip.
- The weights are per image and per channel, not per pixel.

## 6. Soft reconstruction head
- The head predicts K (1 channel) and B (3 channels): **J = I + K·I + B = (1 + K)·I + B**.
- For the haze model I = J·t + A(1 − t), the exact inverse is 1 + K = 1/t, B = −A(1/t − 1). So the head can represent per-pixel haze removal directly.
- The head is zero-initialised, so the untrained network is the identity (J = I).
- Training uses the raw output. The output is clamped to [0, 1] only at test.

---

## 7. Training recipe (SCR_HR, from scratch, seed 42)

| | Phase 1 (700 epochs) | Phase 2 (20 epochs) |
|---|---|---|
| Data | all 6,000 RESIDE-6K training pairs | same |
| Crop / batch | 256² / 8 | 384² / 4 |
| Augmentation | h-flip, scale aug [0.5, 1.55, 1.15] | same + patchy haze, p = 0.25 |
| Optimiser | AdamW 2e-4, wd 1e-4, warmup 8, cosine → 1e-6 | AdamW 1e-4, warmup 1, cosine → 1e-6 |
| Loss | log-MSE | log-MSE |
| Other | EMA 0.999, AMP, adaptive clipping | same |
| Result | 30.7036 / 0.9747 | **31.2128 / 0.9763** |

- Patchy haze: hz' = GT·t^k + A(1 − t^k), with k = 2^u(x) on smooth ~96-px patches, between ½× and 2× density.
- Cost: 144.2 GPU hours on one T4.
- Test: last-epoch EMA weights, no checkpoint selection, full resolution.

---

## 8. Where each component comes from

| Component | Origin | Credit in the paper |
|---|---|---|
| U-Net with stride-2 conv down, PixelShuffle up | standard; same as DehazeFormer | cite |
| RescaleNorm (RLN) | DehazeFormer (TIP 2023) | cite |
| SK fusion of skips | SKNet (CVPR 2019), as used in DehazeFormer | cite |
| Soft reconstruction J = (1 + K)I + B | DehazeFormer (sign of B differs) | cite |
| Dark channel cue | He et al., DCP | cite |
| Per-pixel choice between a small and a large dilated depthwise kernel | LSKNet spatial selection (ICCV 2023) | cite as the closest mechanism |
| **Haze-heterogeneity cues driving the kernel choice** | this work | contribution |
| **Static prior + zero-initialised per-pixel correction, w and 1 − w per group** | this work (idea related to DCD's static + dynamic residual) | contribution |
| **Attention-free design, capacity at 1/8, under 1 M params** | this work | contribution |

---

## 9. Implementations that use this kind of architecture

**HCR-Net itself is unpublished, so no one else has implemented it.** The models below share parts of it. "LSG-Lite" was read as **LSKNet** (Large Selective Kernel); no dehazing model named LSG-Lite turned up in a search.

### 9.1 DehazeFormer (Song et al., IEEE TIP 2023) — the backbone family
- Official code: IDKiro/DehazeFormer (PyTorch).
- Same skeleton: U-Net, RLN, SK fusion, a 4-channel K/B head.

| | DehazeFormer-T / S / B | HCR-Net (SCR_HR) |
|---|---|---|
| Levels | 3 (1, 1/2, 1/4) | 4 (1, 1/2, 1/4, 1/8) |
| Widths | 24, 48, 96, 48, 24 | 24, 48, 96, 136, 96, 48, 24 |
| Depths | T 4,4,4,2,2; S 8,8,8,4,4; B 16,16,16,8,8 | 4,5,2,3,2,5,4 |
| Token mixer | shifted-window attention (window 8) + DW 5×5 conv, summed | DW 3×3 and DW 5×5 d3, routed per pixel |
| Attention blocks | at the end of each of the first 3 layers | none |
| Normalisation | RLN, before attention only | RescaleNorm before both halves |
| Output | x = K·x − B + x | J = I + K·I + B |
| Params | T 0.686 M, S 1.283 M, B 2.51 M | 0.979 M |
| RESIDE-6K PSNR / SSIM | T 30.36 / 0.973; S 30.62 / 0.976; B 31.45 / 0.980; M 30.89 / 0.977 (4.63 M) | **31.21 / 0.976** |

Models that build directly on DehazeFormer:
- **ED-Former (2026):** keeps the DehazeFormer design, adds wavelet resizing, a gated MLP and a staged loss. 0.866 M params, 38.21 dB on SOTS-indoor.
- **CCOP-IDT (J. Computer Applications 2026):** 0.669 M params, 30.09 dB. It re-trains DehazeFormer at 29.99 dB.

### 9.2 LSKNet (Li et al., ICCV 2023) — the closest routing mechanism
- Official code: zcablii/LSKNet (remote-sensing object detection, not dehazing).
- It chooses per pixel between depthwise kernels of different receptive field (DW 5×5 → DW 7×7 d3, in sequence), with sigmoid spatial masks from pooled features.

| | LSKNet | HR in HCR-Net |
|---|---|---|
| Kernels | DW 5×5 d1 → DW 7×7 d3, sequential | DW 3×3 and DW 5×5 d3, parallel |
| Weights | independent sigmoid per kernel | complementary w and 1 − w |
| Granularity | one mask for all channels | 4 channel groups |
| Selector input | avg + max pooled features | features + 6 haze-heterogeneity cues |
| Static start + correction | no | yes |
| Result used as | attention, Y = X · S | mixer output inside a residual block |
| Size | LSKNet-T 4.3 M, S 14.4 M | 0.979 M |
| Static control | no selection: +0.37 mAP, 1 run | parameter-matched static twin, 2 seeds |

### 9.3 Other dehazing models with related parts
| Model | Shared idea | Difference from HCR-Net | Params |
|---|---|---|---|
| HazeTrendNet (Electronics Letters 2025) | dark-channel guide for kernel weighting | fixed DCP formula; w·(Conv3 + Conv5); no static start | 6.17 M |
| DCAF-Net | dark channel + contrast cues → learned haze map | map scales features (F ⊙ W), no kernel trade-off | 43.66 M |
| DGFDNet (arXiv 2507.11035) | dark-channel spatial attention; k = 3 / k = 5 gated branches | gates are feature-driven and do not trade branches | 2.08 M |
| ADPCNet (2026) | per-pixel softmax fusion of local vs large-context branches | gate from branch features, no haze prior | 7.25 M |
| HazeWaveNet (Discover Computing 2026) | DCP trend map, lightweight | modulates low-frequency features, not kernel selection | 0.42 M |
| DEA-Net (TIP 2024) | per-pixel fusion weights (CGA) | only at encoder/decoder fusion | — |
| SKNet (CVPR 2019) | softmax selection between kernel branches | per channel for the whole image | — |

---

## Sources
- [DehazeFormer code (models/dehazeformer.py)](https://github.com/IDKiro/DehazeFormer/blob/main/models/dehazeformer.py)
- [Vision Transformers for Single Image Dehazing (DehazeFormer), arXiv 2204.03883](https://arxiv.org/abs/2204.03883)
- [Large Selective Kernel Network for Remote Sensing Object Detection (LSKNet), arXiv 2303.09030](https://arxiv.org/abs/2303.09030)
- [CCOP-IDT, J. Computer Applications 2026, 46(3)](https://www.joca.cn/EN/Y2026/V46/I3/933)
- [DGFDNet, arXiv 2507.11035](https://arxiv.org/abs/2507.11035)
- Project notes: SCR_HR final result; MHY HR vs static twin; H9 heterogeneity router; HR router novelty check; LSKNet full-text analysis.
